# COCO val2017: 2000 obrazów z dokładnie jedną osobą

In [1]:
from pathlib import Path

COCO_DIR = Path(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_COCO2017_benchmark")
N_IMAGES = 1000
DOWNLOAD_WORKERS = 16

ANNO_DIR = COCO_DIR / "annotations"
IMG_DIR  = COCO_DIR / "images"
GT_JSON_PATH = COCO_DIR / "gt_filtered.json"

for _d in [COCO_DIR, ANNO_DIR, IMG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print("Konfiguracja załadowana.")
print(f"  COCO_DIR:              {COCO_DIR}")
print(f"  N_IMAGES (val only):   {N_IMAGES}")
print(f"  DOWNLOAD_WORKERS:      {DOWNLOAD_WORKERS}")

Konfiguracja załadowana.
  COCO_DIR:              D:\Politechnika\Deadlift_publikacja_nr_2\Dane_COCO2017_benchmark
  N_IMAGES (val only):   1000
  DOWNLOAD_WORKERS:      16


In [2]:
import subprocess, sys, json, zipfile, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def _ensure_package(import_name, pip_name=None):
    """Install package if not available."""
    try:
        __import__(import_name)
    except ImportError:
        pkg = pip_name or import_name
        print(f"Instalowanie: {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  Zainstalowano: {pkg}")

_ensure_package("pycocotools")
_ensure_package("ultralytics")

c:\Users\rwcki\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pobieranie val2017 + filtr 1 osoba

In [3]:
def download_file_with_progress(url: str, dest: Path) -> None:
    """Download url -> dest with tqdm progress. Skip if already exists."""
    if dest.exists():
        print(f"  Plik już istnieje: {dest.name} — pomijanie.")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"  Pobieranie: {url}")
    with tqdm(unit='B', unit_scale=True, unit_divisor=1024, desc=dest.name, leave=False) as pbar:
        def _hook(n, bs, total):
            if total > 0 and pbar.total is None:
                pbar.total = total
            pbar.update(n * bs - pbar.n if pbar.n else n * bs)
        urllib.request.urlretrieve(url, str(dest), _hook)
    print(f"  Zapisano: {dest}")


def prepare_coco_data_val_one_person(coco_dir: Path, n_images: int):
    """
    Build GT only from val2017 and keep images with exactly one person annotation.
    Returns: (filtered_gt_dict, selected_images_list)
    """
    anno_dir = coco_dir / "annotations"
    anno_zip = coco_dir / "annotations_trainval2017.zip"
    val_json_path = anno_dir / "instances_val2017.json"

    if not val_json_path.exists():
        download_file_with_progress(
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
            anno_zip
        )
        print("  Rozpakowywanie anotacji ...")
        with zipfile.ZipFile(str(anno_zip)) as zf:
            zf.extractall(str(coco_dir))
        print("  Gotowe.")

    print("Ładowanie instances_val2017.json ...")
    with open(str(val_json_path), "r") as f:
        val_data = json.load(f)

    person_cat_id = next(c["id"] for c in val_data["categories"] if c["name"] == "person")
    print(f"  person category_id = {person_cat_id}")

    person_anns = [
        a for a in val_data["annotations"]
        if a["category_id"] == person_cat_id and a.get("iscrowd", 0) == 0
    ]

    person_count_by_image = {}
    for ann in person_anns:
        image_id = ann["image_id"]
        person_count_by_image[image_id] = person_count_by_image.get(image_id, 0) + 1

    one_person_ids = sorted([img_id for img_id, cnt in person_count_by_image.items() if cnt == 1])
    print(f"  Val images with exactly one person: {len(one_person_ids)}")

    if len(one_person_ids) < n_images:
        raise RuntimeError(
            f"Za mało obrazów z dokładnie 1 osobą w val2017: {len(one_person_ids)} < {n_images}"
        )

    selected_val_ids = one_person_ids[:n_images]
    print(f"  Wybrano z val2017: {len(selected_val_ids)}")

    val_img_map = {img["id"]: img for img in val_data["images"]}
    selected_images = []
    for img_id in selected_val_ids:
        info = dict(val_img_map[img_id])
        info["_split"] = "val2017"
        selected_images.append(info)

    selected_id_set = {img["id"] for img in selected_images}
    all_annotations = [a for a in person_anns if a["image_id"] in selected_id_set]
    person_category = next(c for c in val_data["categories"] if c["name"] == "person")

    filtered_gt = {
        "info": val_data.get("info", {}),
        "licenses": val_data.get("licenses", []),
        "categories": [person_category],
        "images": [{k: v for k, v in img.items() if not k.startswith("_")} for img in selected_images],
        "annotations": all_annotations,
    }

    print(f"\nŁącznie obrazów GT (val): {len(selected_images)}")
    print(f"Anotacji person:            {len(all_annotations)}")
    return filtered_gt, selected_images


def download_coco_images(selected_images: list, img_dir: Path, n_workers: int = 16) -> list:
    """Download images concurrently. Returns list of failed (img_id, error) tuples."""
    img_dir.mkdir(parents=True, exist_ok=True)

    def _download_one(img_info):
        fname = img_info["file_name"]
        split = img_info.get("_split", "val2017")
        dest = img_dir / fname
        if dest.exists():
            return True, img_info["id"], None
        url = f"http://images.cocodataset.org/{split}/{fname}"
        try:
            urllib.request.urlretrieve(url, str(dest))
            return True, img_info["id"], None
        except Exception as exc:
            return False, img_info["id"], str(exc)

    n_cached = sum(1 for img in selected_images if (img_dir / img["file_name"]).exists())
    n_to_dl = len(selected_images) - n_cached

    if n_to_dl == 0:
        print(f"Wszystkie {len(selected_images)} obrazów już istnieje — pomijanie pobierania.")
        return []

    print(f"Cached: {n_cached} | Do pobrania: {n_to_dl}")
    to_dl = [img for img in selected_images if not (img_dir / img["file_name"]).exists()]

    failed = []
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_download_one, img): img for img in to_dl}
        with tqdm(total=len(to_dl), desc="Pobieranie obrazów") as pbar:
            for fut in as_completed(futures):
                ok, img_id, err = fut.result()
                if not ok:
                    failed.append({"image_id": img_id, "error": err})
                pbar.update(1)

    n_ok = sum(1 for img in selected_images if (img_dir / img["file_name"]).exists())
    print(f"Pobrano. Dostępnych obrazów: {n_ok}/{len(selected_images)}")
    if failed:
        print(f"  Błędy: {len(failed)} plików — sprawdź zmienną failed_downloads.")
    return failed


def filter_with_yolo26x_conf90(selected_images: list, img_dir: Path):
    """
    Keep only images where yolo26x detects at least one person with conf >= 0.90.
    Delete non-matching image files from disk.
    Returns: (kept_images, removed_image_ids, removed_file_names)
    """
    from ultralytics import YOLO
    import torch

    conf_threshold = 0.50
    weights_path = Path.cwd() / "yolo26x.pt"
    if not weights_path.exists() and (Path.cwd() / "scores_from_publication" / "yolo26x.pt").exists():
        weights_path = Path.cwd() / "scores_from_publication" / "yolo26x.pt"

    if not weights_path.exists():
        raise FileNotFoundError(f"Brak wag detektora: {weights_path}")

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = YOLO(str(weights_path))

    kept_images = []
    removed_ids = []
    removed_files = []

    for img in tqdm(selected_images, desc="Filtr YOLO26x conf>=0.90"):
        img_path = img_dir / img["file_name"]
        if not img_path.exists():
            removed_ids.append(img["id"])
            removed_files.append(img["file_name"])
            continue

        has_person = False
        try:
            result = model.predict(
                source=str(img_path),
                conf=conf_threshold,
                verbose=False,
                device=device,
            )[0]
            if result.boxes is not None and len(result.boxes) > 0:
                classes = result.boxes.cls.cpu().numpy().astype(int)
                has_person = bool((classes == 0).any())
        except Exception:
            has_person = False

        if has_person:
            kept_images.append(img)
        else:
            removed_ids.append(img["id"])
            removed_files.append(img["file_name"])
            img_path.unlink(missing_ok=True)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"YOLO26x@0.90 kept: {len(kept_images)} / {len(selected_images)}")
    print(f"YOLO26x@0.90 removed: {len(removed_ids)}")
    return kept_images, set(removed_ids), removed_files

In [4]:
gt_dict, selected_images = prepare_coco_data_val_one_person(COCO_DIR, N_IMAGES)

failed_downloads = download_coco_images(selected_images, IMG_DIR, DOWNLOAD_WORKERS)
failed_ids = {r["image_id"] for r in failed_downloads}
selected_images = [img for img in selected_images if img["id"] not in failed_ids]

# Keep GT aligned with successfully downloaded images first
kept_ids = {img["id"] for img in selected_images}
gt_dict["images"] = [img for img in gt_dict["images"] if img["id"] in kept_ids]
gt_dict["annotations"] = [ann for ann in gt_dict["annotations"] if ann["image_id"] in kept_ids]

# Additional filter: keep only images where yolo26x finds person with conf >= 0.90
selected_images, yolo_removed_ids, yolo_removed_files = filter_with_yolo26x_conf90(selected_images, IMG_DIR)

kept_ids = {img["id"] for img in selected_images}
gt_dict["images"] = [img for img in gt_dict["images"] if img["id"] in kept_ids]
gt_dict["annotations"] = [ann for ann in gt_dict["annotations"] if ann["image_id"] in kept_ids]

print(f"\nZapisywanie przefiltrowanego GT -> {GT_JSON_PATH} ...")
with open(str(GT_JSON_PATH), "w") as _f:
    json.dump(gt_dict, _f)
print("  Gotowe.")

print(f"\nUsunięte przez YOLO26x@0.90: {len(yolo_removed_ids)}")
print(f"Obrazów do ewaluacji po filtrze: {len(selected_images)}")
print(f"Anotacji person po filtrze: {len(gt_dict['annotations'])}")

Ładowanie instances_val2017.json ...
  person category_id = 1
  Val images with exactly one person: 1045
  Wybrano z val2017: 1000

Łącznie obrazów GT (val): 1000
Anotacji person:            1000
Wszystkie 1000 obrazów już istnieje — pomijanie pobierania.


Filtr YOLO26x conf>=0.90:  32%|███▏      | 321/1000 [00:23<00:38, 17.45it/s]

WARNING Image Read Error D:\Politechnika\Deadlift_publikacja_nr_2\Dane_COCO2017_benchmark\images\000000182441.jpg


Filtr YOLO26x conf>=0.90:  35%|███▌      | 350/1000 [00:25<00:38, 16.69it/s]

WARNING Image Read Error D:\Politechnika\Deadlift_publikacja_nr_2\Dane_COCO2017_benchmark\images\000000192964.jpg


Filtr YOLO26x conf>=0.90: 100%|██████████| 1000/1000 [01:05<00:00, 15.25it/s]


YOLO26x@0.90 kept: 911 / 1000
YOLO26x@0.90 removed: 89

Zapisywanie przefiltrowanego GT -> D:\Politechnika\Deadlift_publikacja_nr_2\Dane_COCO2017_benchmark\gt_filtered.json ...
  Gotowe.

Usunięte przez YOLO26x@0.90: 89
Obrazów do ewaluacji po filtrze: 911
Anotacji person po filtrze: 911


In [5]:
failed_downloads

[]

## Weryfikacja zbioru

In [6]:
from collections import Counter
import pandas as pd

# 1) Po filtrze YOLO liczba obrazów moze byc <= N_IMAGES
n_selected = len(selected_images)
print(f"Wybrane obrazy po filtrze YOLO: {n_selected}")
assert n_selected > 0, "Brak obrazów po filtrze YOLO26x@0.90"
assert n_selected <= N_IMAGES, f"Po filtrze jest wiecej niz oczekiwane {N_IMAGES}"

# 2) Czy wszystkie obrazy fizycznie istnieją
missing_files = [img["file_name"] for img in selected_images if not (IMG_DIR / img["file_name"]).exists()]
print(f"Brakujących plików: {len(missing_files)}")
assert len(missing_files) == 0, "Brak niektórych pobranych obrazów"

# 3) Czy GT zawiera dokładnie jedną anotację osoby per image_id
ann_counts = Counter(ann["image_id"] for ann in gt_dict["annotations"])
bad_ids = [img["id"] for img in selected_images if ann_counts.get(img["id"], 0) != 1]
print(f"Image_id z !=1 anotacją: {len(bad_ids)}")
assert len(bad_ids) == 0, "W GT są obrazy bez dokładnie jednej anotacji"

# 4) Szybki podgląd
verify_df = pd.DataFrame([
    {
        "image_id": img["id"],
        "file_name": img["file_name"],
        "n_person_anns": ann_counts.get(img["id"], 0),
    }
    for img in selected_images
])
display(verify_df.head(10))
print("Weryfikacja OK.")

Wybrane obrazy po filtrze YOLO: 911
Brakujących plików: 0
Image_id z !=1 anotacją: 0


,image_id,file_name,n_person_anns
0,785,000000000785.jpg,1
1,1490,000000001490.jpg,1
2,2261,000000002261.jpg,1
3,2532,000000002532.jpg,1
4,3156,000000003156.jpg,1
5,3553,000000003553.jpg,1
6,4395,000000004395.jpg,1
7,4765,000000004765.jpg,1
8,5060,000000005060.jpg,1
9,5529,000000005529.jpg,1


Weryfikacja OK.


## Benchmark segmentacji (mAP50-95, mask)

In [7]:
import time
import gc
import io
import contextlib
import sys as _sys
import numpy as np
import cv2
import torch
import torchvision
import torchvision.transforms.functional as TF
from pycocotools import mask as mask_util
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

_ensure_package("ultralytics")
from ultralytics import YOLO

# Ustal katalog repo dla importu utils
_repo_root = Path.cwd()
if not (_repo_root / "utils").exists() and (_repo_root.parent / "utils").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))

from utils.validation.set_validation import YPSetValidation

# Folder z wagami YOLO-seg
YOLO_WEIGHTS_DIR = Path.cwd()
if not (YOLO_WEIGHTS_DIR / "yolo26n-seg.pt").exists() and (YOLO_WEIGHTS_DIR / "scores_from_publication").exists():
    YOLO_WEIGHTS_DIR = YOLO_WEIGHTS_DIR / "scores_from_publication"

# Wymagane modele YOLO: n, s, m, l, x
YOLO_BENCH_MODELS = ["yolo26n-seg", "yolo26s-seg", "yolo26m-seg", "yolo26l-seg", "yolo26x-seg"]
DETECTION_BENCH_MODELS = ["maskrcnn_resnet50_fpn"]
SEMANTIC_BENCH_MODELS = ["deeplabv3_resnet50", "lraspp_mobilenet_v3_large", "fcn_resnet50"]

missing_yolo_weights = [m for m in YOLO_BENCH_MODELS if not (YOLO_WEIGHTS_DIR / f"{m}.pt").exists()]
if missing_yolo_weights:
    raise FileNotFoundError(
        f"Brak wag YOLO-seg: {missing_yolo_weights}. Uzupełnij pliki .pt w {YOLO_WEIGHTS_DIR}"
    )

print(f"YOLO_WEIGHTS_DIR: {YOLO_WEIGHTS_DIR}")
print(f"YOLO models: {YOLO_BENCH_MODELS}")
print(f"Detection models: {DETECTION_BENCH_MODELS}")
print(f"Semantic models: {SEMANTIC_BENCH_MODELS}")
print(f"Torch: {torch.__version__}, torchvision: {torchvision.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

YOLO_WEIGHTS_DIR: d:\Politechnika\Deadlift_publikacja_nr_2 — kopia\Skrypty\github\scores_from_publication
YOLO models: ['yolo26n-seg', 'yolo26s-seg', 'yolo26m-seg', 'yolo26l-seg', 'yolo26x-seg']
Detection models: ['maskrcnn_resnet50_fpn']
Semantic models: ['deeplabv3_resnet50', 'lraspp_mobilenet_v3_large', 'fcn_resnet50']
Torch: 2.11.0+cu128, torchvision: 0.26.0+cu128
Device: cuda


In [8]:
def _mask_to_rle(binary_mask: np.ndarray) -> dict:
    rle = mask_util.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return rle


def _free_vram() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def predict_yolo_seg(model_name: str, images: list, score_threshold: float = 0.5) -> list:
    weights = YOLO_WEIGHTS_DIR / f"{model_name}.pt"
    model = YOLO(str(weights))
    device = "cuda:0" if torch.cuda.is_available() else "cpu"

    predictions = []
    for img in tqdm(images, desc=f"Inferencja {model_name}"):
        img_path = IMG_DIR / img["file_name"]
        result = model.predict(
            source=str(img_path),
            conf=score_threshold,
            verbose=False,
            device=device,
        )[0]

        if result.masks is None or result.boxes is None:
            continue

        img_h, img_w = result.orig_shape
        classes = result.boxes.cls.cpu().numpy().astype(int)
        scores = result.boxes.conf.cpu().numpy()
        polygons = result.masks.xy

        merged = np.zeros((img_h, img_w), dtype=np.uint8)
        best_score = None

        # Instance-seg: merge multiple person masks into one pseudo-instance per image.
        for cls, score, polygon in zip(classes, scores, polygons):
            if cls != 0 or len(polygon) < 3:
                continue
            comp = np.zeros((img_h, img_w), dtype=np.uint8)
            pts = np.round(polygon).astype(np.int32)
            cv2.fillPoly(comp, [pts], 1)
            merged = np.maximum(merged, comp)
            best_score = float(score) if best_score is None else max(best_score, float(score))

        if merged.max() == 0:
            continue

        predictions.append({
            "image_id": int(img["id"]),
            "category_id": 1,
            "segmentation": _mask_to_rle(merged),
            "score": 1.0 if best_score is None else best_score,
        })

    del model
    _free_vram()
    return predictions


def predict_maskrcnn(images: list, score_threshold: float = 0.5) -> list:
    from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT).to(device).eval()

    predictions = []
    with torch.no_grad():
        for img in tqdm(images, desc="Inferencja maskrcnn_resnet50_fpn"):
            img_path = str(IMG_DIR / img["file_name"])
            img_bgr = cv2.imread(img_path)
            if img_bgr is None:
                continue
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_t = TF.to_tensor(img_rgb).to(device)

            output = model([img_t])[0]
            labels = output["labels"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            masks = output["masks"].cpu().numpy()

            merged = np.zeros(masks.shape[-2:], dtype=np.uint8) if len(masks) > 0 else None
            best_score = None

            # Instance-seg: merge multiple person masks into one pseudo-instance per image.
            for label, score, soft_mask in zip(labels, scores, masks):
                if label != 1 or score < score_threshold:
                    continue
                binary = (soft_mask[0] > 0.5).astype(np.uint8)
                if merged is None:
                    merged = np.zeros_like(binary, dtype=np.uint8)
                merged = np.maximum(merged, binary)
                best_score = float(score) if best_score is None else max(best_score, float(score))

            if merged is None or merged.max() == 0:
                continue

            predictions.append({
                "image_id": int(img["id"]),
                "category_id": 1,
                "segmentation": _mask_to_rle(merged),
                "score": 1.0 if best_score is None else best_score,
            })

    del model
    _free_vram()
    return predictions


def predict_semantic(model_name: str, images: list, min_area: int = 200) -> list:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = YPSetValidation._load_segmentation_model(model_name).to(device).eval()

    voc_person_class = 15
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(3, 1, 1)

    predictions = []
    with torch.no_grad():
        for img in tqdm(images, desc=f"Inferencja {model_name}"):
            img_path = str(IMG_DIR / img["file_name"])
            img_bgr = cv2.imread(img_path)
            if img_bgr is None:
                continue
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_t = TF.to_tensor(img_rgb).to(device)
            img_norm = (img_t - mean) / std

            output = model(img_norm.unsqueeze(0))["out"]
            pred_cls = output.argmax(dim=1).squeeze(0).cpu().numpy()
            person_mask = (pred_cls == voc_person_class).astype(np.uint8)
            if person_mask.max() == 0:
                continue

            n_ccs, labels_map = cv2.connectedComponents(person_mask)
            merged = np.zeros_like(person_mask, dtype=np.uint8)

            # Semantic: multiple components in one image are merged into one pseudo-instance.
            for cc_id in range(1, n_ccs):
                comp = (labels_map == cc_id).astype(np.uint8)
                if comp.sum() < min_area:
                    continue
                merged = np.maximum(merged, comp)

            if merged.max() == 0:
                continue

            predictions.append({
                "image_id": int(img["id"]),
                "category_id": 1,
                "segmentation": _mask_to_rle(merged),
                "score": 1.0,
            })

    del model
    _free_vram()
    return predictions


def compute_coco_map(coco_gt: COCO, predictions: list) -> dict:
    if not predictions:
        return {"AP": 0.0, "AP50": 0.0, "AP75": 0.0, "APs": 0.0, "APm": 0.0, "APl": 0.0}

    gt_img_ids = set(coco_gt.getImgIds())
    preds_filtered = [p for p in predictions if p["image_id"] in gt_img_ids]
    if not preds_filtered:
        return {"AP": 0.0, "AP50": 0.0, "AP75": 0.0, "APs": 0.0, "APm": 0.0, "APl": 0.0}

    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(preds_filtered)

    coco_eval = COCOeval(coco_gt, coco_dt, "segm")
    coco_eval.params.imgIds = sorted(gt_img_ids)
    coco_eval.evaluate()
    coco_eval.accumulate()

    with contextlib.redirect_stdout(io.StringIO()):
        coco_eval.summarize()

    stats = coco_eval.stats
    if stats is None or len(stats) < 6:
        return {"AP": 0.0, "AP50": 0.0, "AP75": 0.0, "APs": 0.0, "APm": 0.0, "APl": 0.0}

    return {
        "AP": float(stats[0]),
        "AP50": float(stats[1]),
        "AP75": float(stats[2]),
        "APs": float(stats[3]),
        "APm": float(stats[4]),
        "APl": float(stats[5]),
    }

In [9]:
with contextlib.redirect_stdout(io.StringIO()):
    coco_gt = COCO(str(GT_JSON_PATH))

benchmark_rows = []
pred_cache = {}

# YOLO-seg
for model_name in YOLO_BENCH_MODELS:
    t0 = time.perf_counter()
    preds = predict_yolo_seg(model_name, selected_images, score_threshold=0.5)
    elapsed = time.perf_counter() - t0

    metrics = compute_coco_map(coco_gt, preds)
    pred_cache[model_name] = preds
    benchmark_rows.append({
        "family": "yolo-seg",
        "model": model_name,
        "n_predictions": len(preds),
        "time_s_total": elapsed,
        "time_ms_per_image": (elapsed / len(selected_images)) * 1000.0,
        **metrics,
    })

# MaskRCNN
for model_name in DETECTION_BENCH_MODELS:
    t0 = time.perf_counter()
    preds = predict_maskrcnn(selected_images, score_threshold=0.5)
    elapsed = time.perf_counter() - t0

    metrics = compute_coco_map(coco_gt, preds)
    pred_cache[model_name] = preds
    benchmark_rows.append({
        "family": "instance-detection",
        "model": model_name,
        "n_predictions": len(preds),
        "time_s_total": elapsed,
        "time_ms_per_image": (elapsed / len(selected_images)) * 1000.0,
        **metrics,
    })

# Modele semantyczne
for model_name in SEMANTIC_BENCH_MODELS:
    t0 = time.perf_counter()
    preds = predict_semantic(model_name, selected_images, min_area=200)
    elapsed = time.perf_counter() - t0

    metrics = compute_coco_map(coco_gt, preds)
    pred_cache[model_name] = preds
    benchmark_rows.append({
        "family": "semantic-seg",
        "model": model_name,
        "n_predictions": len(preds),
        "time_s_total": elapsed,
        "time_ms_per_image": (elapsed / len(selected_images)) * 1000.0,
        **metrics,
    })

benchmark_df = pd.DataFrame(benchmark_rows).sort_values("AP", ascending=False).reset_index(drop=True)
display(benchmark_df)

print("\nTop-3 modele po AP:")
display(benchmark_df.head(3))

Inferencja yolo26n-seg: 100%|██████████| 911/911 [00:38<00:00, 23.92it/s]


Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.41s).
Accumulating evaluation results...
DONE (t=0.07s).


Inferencja yolo26s-seg: 100%|██████████| 911/911 [00:39<00:00, 23.34it/s]


Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.31s).
Accumulating evaluation results...
DONE (t=0.08s).


Inferencja yolo26m-seg: 100%|██████████| 911/911 [00:42<00:00, 21.50it/s]


...

Inferencja fcn_resnet50: 100%|██████████| 911/911 [00:40<00:00, 22.66it/s]


Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.22s).
Accumulating evaluation results...
DONE (t=0.06s).


,family,model,n_predictions,time_s_total,time_ms_per_image,AP,AP50,AP75,APs,APm,APl
0,yolo-seg,yolo26x-seg,888,60.728899,66.661799,0.690874,0.947462,0.822158,0.297660,0.586327,0.743369
1,yolo-seg,yolo26l-seg,877,46.593890,51.145873,0.684333,0.936675,0.821898,0.265578,0.583466,0.741800
2,yolo-seg,yolo26m-seg,874,42.881836,47.071170,0.678351,0.938301,0.814053,0.306493,0.557988,0.738227
3,yolo-seg,yolo26s-seg,861,39.465581,43.321165,0.656324,0.917537,0.782000,0.260497,0.525817,0.720250
4,instance-detection,maskrcnn_resnet50_fpn,897,108.197151,118.767454,0.619495,0.923498,0.722809,0.393437,0.522136,0.670733
5,yolo-seg,yolo26n-seg,796,38.940919,42.745246,0.609595,0.848675,0.723633,0.178978,0.454961,0.684490
6,semantic-seg,deeplabv3_resnet50,874,45.893925,50.377525,0.529992,0.854038,0.600764,0.069581,0.350796,0.651156
7,semantic-seg,fcn_resnet50,886,41.106638,45.122545,0.523669,0.845508,0.571253,0.091399,0.342674,0.644420
8,semantic-seg,lraspp_mobilenet_v3_large,848,24.356264,26.735745,0.385318,0.755793,0.347280,0.009798,0.186701,0.518278



Top-3 modele po AP:


,family,model,n_predictions,time_s_total,time_ms_per_image,AP,AP50,AP75,APs,APm,APl
0,yolo-seg,yolo26x-seg,888,60.728899,66.661799,0.690874,0.947462,0.822158,0.297660,0.586327,0.743369
1,yolo-seg,yolo26l-seg,877,46.593890,51.145873,0.684333,0.936675,0.821898,0.265578,0.583466,0.741800
2,yolo-seg,yolo26m-seg,874,42.881836,47.071170,0.678351,0.938301,0.814053,0.306493,0.557988,0.738227
